# Retail Analytics Data Platform

## Layer: Bronze

### Objective
Ingest raw Online Retail II dataset into Delta Lake Bronze Layer.

### Activities
- Schema Enforcement
- Raw Data Ingestion
- Data Profiling
- Data Quality Assessment
- Bronze Delta Table Creation

### Source
Online Retail II Dataset

### Target Table
workspace.retail.bronze_online_retail

In [0]:
# Read from CSV file in volume
from pyspark.sql.types import *
from pyspark.sql import functions as F

retail_schema = StructType([
    StructField("Invoice", StringType(), True),
    StructField("StockCode", StringType(), True),
    StructField("Description", StringType(), True),
    StructField("Quantity", IntegerType(), True),
    StructField("InvoiceDate", StringType(), True),
    StructField("Price", DoubleType(), True),
    StructField("Customer ID", StringType(), True),
    StructField("Country", StringType(), True)
])

In [0]:
df = spark.read \
    .format("csv") \
    .option("header", "true") \
    .schema(retail_schema) \
    .load("/Volumes/workspace/retail/retail_analyzer/online_retail_II.csv")

In [0]:
try:
    print(f"Total Records : {df.count()}")
    print(f"Total Columns : {len(df.columns)}")
except Exception as e:
    print(f"Error: Unable to read DataFrame. {str(e)}")
    print("\nPlease verify:")
    print("1. The table 'workspace.retail.retail_online_II' exists")
    print("2. Cell 2 loaded the correct table name")

Total Records : 1067371
Total Columns : 8


In [0]:
df.printSchema()

root
 |-- Invoice: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: string (nullable = true)
 |-- Price: double (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Country: string (nullable = true)



In [0]:
display(df.limit(10))

Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
489434,22041,"""RECORD FRAME 7"""" SINGLE SIZE """,48,2009-12-01 07:45:00,2.1,13085.0,United Kingdom
489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01 07:45:00,1.65,13085.0,United Kingdom
489434,21871,SAVE THE PLANET MUG,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01 07:45:00,5.95,13085.0,United Kingdom
489435,22350,CAT BOWL,12,2009-12-01 07:46:00,2.55,13085.0,United Kingdom
489435,22349,"DOG BOWL , CHASING BALL DESIGN",12,2009-12-01 07:46:00,3.75,13085.0,United Kingdom


In [0]:
null_analysis = df.select([
    F.count(
        F.when(F.col(c).isNull(), c)
    ).alias(c)
    for c in df.columns
])

display(null_analysis)

Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,0,4382,0,0,0,243007,0


In [0]:
total_rows = df.count()

distinct_rows = df.distinct().count()

duplicate_rows = total_rows - distinct_rows

print(f"Total Rows     : {total_rows}")
print(f"Distinct Rows  : {distinct_rows}")
print(f"Duplicate Rows : {duplicate_rows}")

Total Rows     : 1067371
Distinct Rows  : 1033036
Duplicate Rows : 34335


In [0]:
country_distribution = (
    df.groupBy("Country")
      .count()
      .orderBy(F.desc("count"))
)

display(country_distribution)

Country,count
United Kingdom,981330
EIRE,17866
Germany,17624
France,14330
Netherlands,5140
Spain,3811
Switzerland,3189
Belgium,3123
Portugal,2620
Australia,1913


In [0]:
## Adding meta data

bronze_df = (
    df.withColumn(
        "ingestion_timestamp",
        F.current_timestamp()
    )
    .withColumn(
        "source_file",
        F.lit("online_retail_II.csv")
    )
)

Why did you add metadata columns in Bronze Layer?

Answer:

Metadata columns such as ingestion_timestamp and source_file improve auditability, lineage tracking, troubleshooting, and operational monitoring. They help identify when data was loaded and from which source it originated, making the pipeline more reliable and maintainable.

In [0]:
(
    bronze_df.withColumnRenamed("Customer ID", "Customer_ID")
             .write
             .format("delta")
             .mode("overwrite")
             .saveAsTable(
                 "workspace.retail.bronze_online_retail"
             )
)

In [0]:
bronze_df = spark.table(
    "workspace.retail.bronze_online_retail"
)

display(bronze_df.limit(10))

Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer_ID,Country,ingestion_timestamp,source_file
489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,2026-06-14T07:12:40.627Z,online_retail_II.csv
489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,2026-06-14T07:12:40.627Z,online_retail_II.csv
489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,2026-06-14T07:12:40.627Z,online_retail_II.csv
489434,22041,"""RECORD FRAME 7"""" SINGLE SIZE """,48,2009-12-01 07:45:00,2.1,13085.0,United Kingdom,2026-06-14T07:12:40.627Z,online_retail_II.csv
489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,2026-06-14T07:12:40.627Z,online_retail_II.csv
489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01 07:45:00,1.65,13085.0,United Kingdom,2026-06-14T07:12:40.627Z,online_retail_II.csv
489434,21871,SAVE THE PLANET MUG,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,2026-06-14T07:12:40.627Z,online_retail_II.csv
489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01 07:45:00,5.95,13085.0,United Kingdom,2026-06-14T07:12:40.627Z,online_retail_II.csv
489435,22350,CAT BOWL,12,2009-12-01 07:46:00,2.55,13085.0,United Kingdom,2026-06-14T07:12:40.627Z,online_retail_II.csv
489435,22349,"DOG BOWL , CHASING BALL DESIGN",12,2009-12-01 07:46:00,3.75,13085.0,United Kingdom,2026-06-14T07:12:40.627Z,online_retail_II.csv


In [0]:
source_count = df.count()

bronze_count = bronze_df.count()

print(f"Source Count : {source_count}")
print(f"Bronze Count : {bronze_count}")

assert source_count == bronze_count

Source Count : 1067371
Bronze Count : 1067371


Why Bronze Layer?

Bronze layer stores raw source data with minimal modifications. It acts as the immutable source of truth and supports auditing, lineage tracking, and reprocessing.

Why Explicit Schema?

Explicit schema eliminates the additional scan required by inferSchema, improves performance, and ensures schema consistency across ingestion runs.

Why Delta Format?

Delta Lake provides ACID transactions, schema enforcement, schema evolution, time travel, and better reliability compared to traditional Parquet storage.